# 🎼 Módulo 17 — Agentes en Workflows / Orchestrations (resumen)

> **Objetivo:** Repaso integrado de cómo se integran agentes con el framework de workflows, incluyendo el patrón de bajo nivel con `WorkflowBuilder + add_edge`.

> 📘 **Si quieres profundizar en cada orchestration**, ve los notebooks dedicados:
> - [Módulo 09 — Sequential](./09_orchestration_sequential.ipynb)
> - [Módulo 10 — Concurrent](./10_orchestration_concurrent.ipynb)
> - [Módulo 11 — Handoff](./11_orchestration_handoff.ipynb)
> - [Módulo 12 — Group Chat](./12_orchestration_groupchat.ipynb)

## ¿Por qué usar la API de orchestrations?

El **Módulo 16** coordinaba agentes manualmente con `agent.run()`. Funciona, pero:
- ❌ Sin observabilidad (no hay eventos centralizados)
- ❌ Sin agregación automática para fan-out
- ❌ Hay que escribir el código de coordinación cada vez

La API de `agent_framework.orchestrations` te da bloques pre-construidos:

| Builder | Patrón | Cuándo usar |
|---------|--------|-------------|
| `SequentialBuilder` | Pipeline secuencial | A → B → C |
| `ConcurrentBuilder` | Fan-out + agregación | múltiples opiniones |
| `HandoffBuilder` | Delegación dinámica | atención multi-especialista |
| `GroupChatBuilder` | Conversación grupal | debate, brainstorming |

Y para casos avanzados puedes usar `WorkflowBuilder` directamente con agentes como nodos.

## Comparación con C#

| C# | Python |
|----|--------|
| `AgentWorkflowBuilder.BuildSequential(a, b)` | `SequentialBuilder(participants=[a, b]).build()` |
| `AgentWorkflowBuilder.BuildConcurrent([a, b])` | `ConcurrentBuilder(participants=[a, b]).build()` |
| `AgentWorkflowBuilder.CreateGroupChatBuilderWith(...)` | `GroupChatBuilder(participants=[...], ...).build()` |


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Pipeline secuencial con `SequentialBuilder`

Equivalente al pipeline manual del Módulo 12 pero gestionado por el framework.
La conversación fluye entre participantes; cada uno agrega su respuesta al contexto.


In [ ]:
from agent_framework import AgentResponse, Message
from agent_framework.orchestrations import SequentialBuilder

writer = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un escritor creativo. Escribe exactamente 2 oraciones sobre el tema "
        "proporcionado. Responde solo con el texto."
    ),
    name="Escritor",
)

translator = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un traductor profesional. Traduce el texto completo al inglés. "
        "Responde solo con la traducción."
    ),
    name="Traductor",
)

workflow = SequentialBuilder(participants=[writer, translator]).build()

events = await workflow.run("La exploración espacial y el futuro de la humanidad")

conversation = [Message(role="user", contents=["La exploración espacial y el futuro de la humanidad"])]
for output in events.get_outputs():
    # outputs son list[Message] o AgentResponse según orchestrator
    if isinstance(output, list):
        conversation.extend(output)
    elif hasattr(output, 'messages'):
        conversation.extend(output.messages)

print("🔄 Sequential workflow (Escritor → Traductor):\n")
for i, msg in enumerate(conversation, 1):
    name = msg.author_name or msg.role
    preview = (msg.text or "")[:200]
    print(f"   {i:02d} [{name}]\n     {preview}\n")


## 2️⃣ Fan-out paralelo con `ConcurrentBuilder`

Equivale al fan-out con `asyncio.gather` pero el framework también **agrega** los resultados.
El output es un `list[Message]` con la conversación completa de todos los participantes.


In [ ]:
from agent_framework.orchestrations import ConcurrentBuilder

optimist = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un analista extremadamente optimista. Analiza el tema proporcionado "
        "en una oración destacando solo aspectos positivos. Responde en español."
    ),
    name="Optimista",
)

pessimist = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un analista muy pesimista. Analiza el tema proporcionado en una oración "
        "señalando solo riesgos y problemas. Responde en español."
    ),
    name="Pesimista",
)

workflow = ConcurrentBuilder(participants=[optimist, pessimist]).build()

events = await workflow.run(
    "El impacto de la inteligencia artificial en el mercado laboral"
)

print("⚡ Concurrent workflow (Optimista ↔ Pesimista):\n")
outputs = events.get_outputs()
if outputs:
    messages = outputs[0] if isinstance(outputs[0], list) else [outputs[0]]
    for i, msg in enumerate(messages, 1):
        name = msg.author_name or msg.role
        preview = (msg.text or "")[:200]
        print(f"   {i:02d} [{name}]\n     {preview}\n")


## 3️⃣ Group Chat round-robin con `GroupChatBuilder`

Tres roles debaten un tema por turnos. Configurable con:
- `selection_func` — quién habla a continuación
- `termination_condition` — cuándo terminar el debate


In [ ]:
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

developer = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un desarrollador de software. En las discusiones, aporta la perspectiva "
        "técnica sobre viabilidad e implementación. Responde en 1-2 oraciones concisas en español."
    ),
    name="Desarrollador",
)

designer = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un diseñador UX/UI. En las discusiones, aporta la perspectiva de "
        "experiencia de usuario y diseño visual. Responde en 1-2 oraciones concisas en español."
    ),
    name="Disenador",
)

product_manager = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un gerente de producto. En las discusiones, aporta la perspectiva de "
        "negocio, priorización y valor para el cliente. Responde en 1-2 oraciones concisas en español."
    ),
    name="GerenteProducto",
)


def round_robin(state: GroupChatState) -> str:
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]


workflow = (
    GroupChatBuilder(
        participants=[developer, designer, product_manager],
        # 1 ronda = 3 mensajes (uno por participante). Limitar a 6 ⇒ 2 rondas.
        termination_condition=lambda conv: len(conv) >= 6,
        selection_func=round_robin,
    )
    .build()
)

events = await workflow.run("¿Deberíamos agregar un modo oscuro a nuestra aplicación móvil?")

print("🗣️ Group Chat (Round-Robin, 2 rondas):\n")
outputs = events.get_outputs()
if outputs:
    messages = outputs[0] if isinstance(outputs[0], list) else [outputs[0]]
    for i, msg in enumerate(messages, 1):
        name = msg.author_name or msg.role
        preview = (msg.text or "")[:200]
        print(f"   {i:02d} [{name}]\n     {preview}\n")


## 4️⃣ Workflow manual con agentes como nodos

Para casos donde necesitas control total del grafo, los agentes pueden usarse directamente
como ejecutores en `WorkflowBuilder` (es el equivalente Python de `BindAsExecutor` en C#).


In [ ]:
from agent_framework import WorkflowBuilder

researcher = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un investigador. Dado un tema, escribe exactamente 2 datos curiosos "
        "sobre él. Responde solo con los datos."
    ),
    name="Investigador",
)

summarizer = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un editor. Toma el texto proporcionado y genera un resumen de una sola "
        "oración que capture la esencia. Responde solo con el resumen."
    ),
    name="ResumidorEditor",
)

workflow = (
    WorkflowBuilder(start_executor=researcher)
    .add_edge(researcher, summarizer)
    .build()
)

events = await workflow.run("Los pulpos y su inteligencia")

print("🔧 Workflow manual con add_edge (Investigador → Resumidor):\n")
for output in events.get_outputs():
    if isinstance(output, AgentResponse):
        msg = output.messages[0]
        preview = (output.text or "")[:300]
        print(f"   [{msg.author_name}]\n     {preview}\n")


## 🎯 Resumen

| API | Cuándo usar |
|-----|-------------|
| `SequentialBuilder` | Pipeline lineal de agentes |
| `ConcurrentBuilder` | Fan-out paralelo con agregación |
| `GroupChatBuilder` | Conversación grupal con selector configurable |
| `WorkflowBuilder` + agentes | Topología arbitraria con edges (condicionales también) |

## 🎓 ¡Workshop completo!

Has recorrido los 14 módulos del workshop:

| # | Tema |
|---|------|
| 00 | Creación básica de agentes |
| 01 | Ejecución (completa / streaming / opciones) |
| 02 | Salida estructurada con Pydantic |
| 03 | Function tools |
| 04 | Tool approval (HITL) |
| 05 | Multimodal (imágenes) |
| 06 | Conversaciones y sesiones |
| 07 | Context providers |
| 08 | Middleware |
| 09 | Workflows: ejecutores |
| 10 | Workflows: edges condicionales |
| 11 | Workflows: eventos y loops |
| 12 | Múltiples agentes manual |
| 13 | Agentes en workflows (este) |

## 📚 Siguiente paso

Explora los samples oficiales del SDK:
- 🐍 [agent-framework Python samples](https://github.com/microsoft/agent-framework/tree/main/python/samples)
- 📖 [Workshop equivalente en C#](../../01-AgentFrameworkTests/WORKSHOP.md)

¡Feliz coding! 🚀
